# Notebook 16 — Biopython Fundamentals

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 16 of 20  
**Prerequisites:** Notebook 15  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Biopython is the foundational library for biological sequence analysis in Python.
It provides parsers for FASTA/FASTQ/GenBank, wrappers for BLAST, access to NCBI
Entrez, and sequence objects with built-in biological operations.

It is a prerequisite for Modules 08 (Bioinformatics & Sequence Analysis) and 09
(Genomics & NGS). This notebook covers the core objects and I/O.

---
## Step 2 — Intuition

A `SeqRecord` is a FASTA entry: a sequence plus a name and annotations. A `Seq` is
the sequence itself — it behaves like a string but has biological operations built in
(reverse complement, translation, complement). The `SeqIO` module reads and writes
all standard biological sequence file formats.

---
## Step 3 — Biological Background

**FASTA format:**
```
>NM_007294 BRCA1 gene, exon 1
ATGCGATCGATCGATCG...
>NM_000546 TP53 gene, exon 1
ATGGAGGAGCCGCAGT...
```
The `>` line is the header; subsequent lines are the sequence. Multi-line sequences
are common. SeqIO handles multi-line FASTA correctly; a naive `split('>')` does not.

**The genetic code:**  
64 codons map to 20 amino acids + stop codons. The standard (NCBI) table is
`CodonTable.standard_dna_table`. Mitochondrial organisms use different tables.

---
## Step 4 — Mathematical Explanation

**GC skew** is a measure of strand bias:
$$\text{GC skew} = \frac{G - C}{G + C}$$

Positive values indicate more G than C on the given strand; this correlates with
replication direction in bacterial chromosomes. It is computed in sliding windows
across a genome.

---
## Step 5 — Computational Explanation

| Object | Purpose | Example |
|--------|---------|--------|
| `Seq` | Biological sequence string | `Seq("ATGCATGC")` |
| `SeqRecord` | Sequence + ID + metadata | `SeqIO.read(file, "fasta")` |
| `SeqIO.parse` | Iterate over multi-record files | Generator — lazy |
| `SeqIO.write` | Write SeqRecord list to file | `SeqIO.write(records, out, "fasta")` |
| `Entrez` | Access NCBI databases | Requires email |
| `pairwise2` / `Align` | Pairwise sequence alignment | Smith-Waterman / Needleman-Wunsch |

---
## Step 6 — Python Implementation

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import io, pathlib

print(f"Biopython version:")
import Bio; print(Bio.__version__)

In [ ]:
# Cell 6.1 — Seq: basic operations
dna = Seq("ATGCGATCGATCGATCG")
print(f"Sequence:          {dna}")
print(f"Length:            {len(dna)}")
print(f"Complement:        {dna.complement()}")
print(f"Reverse complement:{dna.reverse_complement()}")
print(f"GC content:        {sum(1 for nt in dna if nt in 'GC') / len(dna):.2%}")

# Slicing works like a string
print(f"First codon:       {dna[:3]}")
print(f"Reading frame 0:   {dna[0::3]}")

In [ ]:
# Cell 6.2 — Translation
coding_seq = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")
protein = coding_seq.translate()
protein_no_stop = coding_seq.translate(to_stop=True)
print(f"CDS:          {coding_seq}")
print(f"Protein:      {protein}")
print(f"No stop:      {protein_no_stop}")
print(f"Starts with M: {str(protein_no_stop)[0] == 'M'}")

In [ ]:
# Cell 6.3 — SeqRecord and SeqIO: read/write FASTA
fasta_content = """>BRCA1_exon1 Homo sapiens BRCA1 gene
ATGCGATCGATCGATCGATCGATCGATCGATCG
ATCGATCGATCGATCG
>TP53_exon1 Homo sapiens TP53 gene
ATGGAGGAGCCGCAGTCAGATCCTAGCGAGCAG
"""

records = list(SeqIO.parse(io.StringIO(fasta_content), "fasta"))
print(f"Parsed {len(records)} records")

for rec in records:
    print(f"  ID: {rec.id:<20}  len={len(rec.seq)}  GC={sum(1 for nt in rec.seq if nt in 'GC')/len(rec.seq):.1%}")

In [ ]:
# Cell 6.4 — Add read_fasta / write_fasta to compbio_utils/io.py
io_content = '''
"""I/O utilities for biological sequence and expression data."""
from __future__ import annotations
import io
from pathlib import Path
from typing import Iterator

from Bio import SeqIO
from Bio.SeqRecord import SeqRecord


def read_fasta(path: str | Path) -> list[SeqRecord]:
    """
    Read a FASTA file and return a list of SeqRecord objects.

    Parameters
    ----------
    path : str or Path
        Path to the FASTA file.

    Returns
    -------
    list of SeqRecord

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"FASTA file not found: {path}")
    return list(SeqIO.parse(path, "fasta"))


def iter_fasta(path: str | Path) -> Iterator[SeqRecord]:
    """
    Lazily iterate over records in a FASTA file (memory-efficient for large files).
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"FASTA file not found: {path}")
    yield from SeqIO.parse(path, "fasta")
'''

import pathlib
repo_root = pathlib.Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists(): break
    repo_root = repo_root.parent

io_path = repo_root / "utilities" / "compbio_utils" / "io.py"
if not io_path.exists():
    io_path.write_text(io_content.strip())
    print(f"Written: {io_path}")
else:
    print(f"io.py exists — merge read_fasta and iter_fasta if not present: {io_path}")

In [ ]:
# Cell 6.5 — GC skew across a genome window
def gc_skew(sequence: str, window: int = 1000) -> list[float]:
    """GC skew (G-C)/(G+C) in sliding windows."""
    skews = []
    for i in range(0, len(sequence) - window + 1, window):
        chunk = sequence[i:i+window].upper()
        g = chunk.count("G")
        c = chunk.count("C")
        skews.append((g - c) / (g + c + 1e-9))
    return skews

# Test on EcoRI + flanking
test_seq = "GCGCGCGCGCGCATATATATATATATAT" * 4
skews = gc_skew(test_seq, window=14)
print("GC skew per window:", [round(s, 3) for s in skews])

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — GC skew plot on synthetic sequence
import matplotlib.pyplot as plt
import numpy as np

# Synthetic genome: GC-rich left half, AT-rich right half
rng = np.random.default_rng(42)
genome = (
    "".join(rng.choice(list("GCGCGCGGCA"), size=5000))
    + "".join(rng.choice(list("ATATATAATA"), size=5000))
)
w = 200
skews = gc_skew(genome, window=w)
positions = list(range(0, len(genome) - w + 1, w))

fig, ax = plt.subplots(figsize=(10, 3))
ax.fill_between(positions, skews, 0,
                where=[s > 0 for s in skews], color="tomato", alpha=0.7, label="G > C")
ax.fill_between(positions, skews, 0,
                where=[s < 0 for s in skews], color="steelblue", alpha=0.7, label="C > G")
ax.axhline(0, color="black", lw=0.5)
ax.set_xlabel("Genomic position")
ax.set_ylabel("GC skew")
ax.set_title(f"GC skew (window={w}bp) — synthetic genome")
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Write `write_fasta(records, path)` in `compbio_utils/io.py` and add tests.
2. Use `Seq.translate()` to find all three reading frames of BRCA1 exon 1.
   Which frame produces the longest ORF before a stop codon?
3. Add `gc_skew` to `compbio_utils/sequence.py`.
4. Using `Bio.Entrez`, fetch the FASTA sequence for NM_007294.4 (BRCA1 mRNA).
   Save it to `datasets/raw/brca1_nm007294.fasta` (do not commit the file — add to
   `.gitignore`).

---
## Quiz — Active Recall

1. What is the difference between `SeqIO.parse` and `SeqIO.read`? When does each fail?
2. Why does `iter_fasta` use `yield from` instead of `return list(...)`?
3. What does `translate(to_stop=True)` do differently from `translate()`?
4. What is GC skew? In which biological context is it most diagnostic?
5. What is the difference between DNA, RNA, and protein `Seq` objects in Biopython?

---
## Reflection

**Date completed:** ____________________

> *[Is read_fasta in compbio_utils with tests? Did the Entrez fetch work? What surprised you about Biopython?]*

---
*Next: `17_unix_cli_bioinformatics.ipynb`*